# A correção de 2026: MERGE no Delta Lake

## Task 4: corrigir os placares de 2026 sem recarregar a Copa

Nas tabelas do vídeo 5.3, 2026 aparece com **333 gols**; o oficial é **308**. Nas partidas decididas nos pênaltis,
a fonte de 2026 soma os gols da disputa no placar. O fornecedor mandou um **lote de correção** em
`data/raw_2026_correcao`, com as 4 partidas.

In [1]:
import sys
sys.path.insert(0, "../../src")

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession, functions as F

construtor = (
    SparkSession.builder
    .appName("arena-correcao")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(construtor).getOrCreate()

RAW = "../../data/raw"
RAW_CORRECAO = "../../data/raw_2026_correcao"
OUT = "../../data/out"

spark

A célula abaixo já vem pronta: a leitura e a gravação de `src/wc_pipeline/carga.py` (vídeo 5.3) e as funções do resumo.

In [2]:
# vídeo 5.3: a leitura e a gravação, no projeto
from wc_pipeline.carga import ler_partidas, gravar
from wc_pipeline.transformacoes import partidas_por_selecao, resumo_por_selecao

## O placar de 2026 hoje

In [3]:
partidas = spark.read.format("delta").load(f"{OUT}/partidas")
partidas.filter(F.col("ano") == 2026).select(F.sum("total_gols")).show()

+---------------+
|sum(total_gols)|
+---------------+
|            333|
+---------------+



Agora o lote de correção, com a mesma `ler_partidas`: ela recalcula `total_gols` e `resultado`.

In [5]:
correcao = ler_partidas(spark, RAW_CORRECAO)
correcao.select("match_id", "fase", "mandante", "gols_mandante", "gols_visitante", "visitante", "total_gols", "resultado").show()

+--------+-----------+-----------+-------------+--------------+---------+----------+---------+
|match_id|       fase|   mandante|gols_mandante|gols_visitante|visitante|total_gols|resultado|
+--------+-----------+-----------+-------------+--------------+---------+----------+---------+
|  537382|Round of 16|Switzerland|            0|             0| Colombia|         0|   empate|
|  537415|Round of 32|    Germany|            1|             1| Paraguay|         2|   empate|
|  537418|Round of 32|Netherlands|            1|             1|  Morocco|         2|   empate|
|  537428|Round of 32|  Australia|            1|             1|    Egypt|         2|   empate|
+--------+-----------+-----------+-------------+--------------+---------+----------+---------+



## Por que não o overwrite?

O overwrite dinâmico troca a **pasta inteira** de 2026: ela ficaria só com as 4 partidas do lote. O lote é **parcial**:
é o caso do `MERGE`.

## MERGE INTO

| A chave do lote... | O que o MERGE faz | Como se escreve |
|---|---|---|
| já existe na tabela | atualiza a linha | `WHEN MATCHED THEN UPDATE SET *` |
| não existe na tabela | insere a linha | `WHEN NOT MATCHED THEN INSERT *` |

A chave é o `match_id`.

In [7]:
from pathlib import Path

correcao.createOrReplaceTempView("correcao")
caminho = Path(f"{OUT}/partidas").resolve()

spark.sql(f"""
    MERGE INTO delta.`{caminho}` AS destino
    USING correcao AS lote
    ON destino.match_id = lote.match_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""").show()

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|                4|               4|               0|                0|
+-----------------+----------------+----------------+-----------------+



4 linhas atualizadas, nenhuma inserida.

## Conferindo a correção

In [8]:
partidas = spark.read.format("delta").load(f"{OUT}/partidas")

print(partidas.count(), "partidas")
partidas.filter(F.col("ano") == 2026).select(F.sum("total_gols")).show()

1068 partidas
+---------------+
|sum(total_gols)|
+---------------+
|            308|
+---------------+



In [9]:
(
    partidas.join(correcao.select("match_id"), on="match_id")
    .select("mandante", "gols_mandante", "gols_visitante", "visitante", "resultado")
    .show()
)

+-----------+-------------+--------------+---------+---------+
|   mandante|gols_mandante|gols_visitante|visitante|resultado|
+-----------+-------------+--------------+---------+---------+
|Switzerland|            0|             0| Colombia|   empate|
|    Germany|            1|             1| Paraguay|   empate|
|Netherlands|            1|             1|  Morocco|   empate|
|  Australia|            1|             1|    Egypt|   empate|
+-----------+-------------+--------------+---------+---------+



## O resumo por seleção

O resumo é uma **agregação**: recalcule 2026 a partir das partidas corrigidas e grave só 2026, com o overwrite
dinâmico da função `gravar`.

In [10]:
resumo = spark.read.format("delta").load(f"{OUT}/resumo_selecoes")
resumo.show(3)

+----+-------+-----+--------+-------+--------+--------+-----------+-----+
| ano|selecao|jogos|vitorias|empates|derrotas|gols_pro|gols_contra|saldo|
+----+-------+-----+--------+-------+--------+--------+-----------+-----+
|1990|Belgium|    4|       2|      0|       2|       6|          4|    2|
|1990|Austria|    3|       1|      0|       2|       2|          3|   -1|
|1990| Sweden|    3|       0|      0|       3|       3|          6|   -3|
+----+-------+-----+--------+-------+--------+--------+-----------+-----+
only showing top 3 rows


In [11]:
partidas_2026 = partidas.filter(F.col("ano") == 2026)
resumo_corrigido = resumo_por_selecao(partidas_por_selecao(partidas_2026))

gravar(resumo_corrigido, f"{OUT}/resumo_selecoes")

In [12]:
resumo = spark.read.format("delta").load(f"{OUT}/resumo_selecoes")
resumo.filter((F.col("ano") == 2026) & F.col("selecao").isin("Germany", "Paraguay")).show()

+----+--------+-----+--------+-------+--------+--------+-----------+-----+
| ano| selecao|jogos|vitorias|empates|derrotas|gols_pro|gols_contra|saldo|
+----+--------+-----+--------+-------+--------+--------+-----------+-----+
|2026|Paraguay|    5|       1|      2|       2|       3|          6|   -3|
|2026| Germany|    4|       2|      1|       1|      11|          5|    6|
+----+--------+-----+--------+-------+--------+--------+-----------+-----+



## Resultado da task 4

2026 corrigida nas duas tabelas, sem recarregar a Copa.

## Demanda da aula 5 atendida

| Etapa | Entrega |
|---|---|
| Task 1 | `overwrite` em Delta Lake |
| Task 2 | saída por Copa, com partition pruning |
| Task 3 | overwrite dinâmico e as tabelas gravadas |
| Task 4 | `MERGE INTO` nas partidas e o resumo de 2026 recalculado |

## Onde cada coisa foi parar

| Aula | Onde ficou | Como é usada a partir de agora |
|---|---|---|
| 1 | `src/wc_pipeline/schemas.py` | cada lote novo é lido com o mesmo schema |
| 2 | `src/wc_pipeline/transformacoes.py` | as cargas e as correções passam pelas mesmas funções |
| 3 | `sql/gols_por_copa.sql` | analytics roda a consulta sobre as partidas consolidadas |
| 4 | `data/out/passes_*`, em Parquet | estudo pontual, refeito só se pedirem |
| 5 | `src/wc_pipeline/carga.py` e as tabelas `data/out/partidas` e `data/out/resumo_selecoes`, em Delta | Copa nova: `carregar_copa`, com overwrite dinâmico; correção: `MERGE` |

## Para consultar depois

- **O histórico da correção:** como no vídeo 5.1, o `DESCRIBE HISTORY` da tabela de partidas mostra a correção como
  uma versão nova, com a operação `MERGE`, e o `versionAsOf` lê a versão anterior, com os 333 gols.
- **Mais condições no `MERGE`:** `WHEN MATCHED AND <condição> THEN ...` atualiza só parte das linhas, `WHEN MATCHED THEN
  DELETE` apaga, e `WHEN NOT MATCHED BY SOURCE` trata as linhas da tabela que não vieram no lote (docs.delta.io, Table
  deletes, updates, and merges).
- **`MERGE` em Python:** `DeltaTable.forPath(spark, caminho).merge(...)` faz o mesmo sem SQL.
- **Arquivos das versões antigas:** ficam na pasta até um `VACUUM`, que apaga os que não fazem mais parte das versões
  recentes (vídeo 5.1).
- **A carga sem notebook:** na empresa, um script `.py` importa o `carregar_copa` e é executado com `spark-submit` (com
  `--packages` para o Delta) e chamado por um agendador, como o Airflow, a cada lote. A atividade "Subindo o projeto no
  Git" traz esse passo como opcional.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [13]:
spark.stop()